<a href="https://colab.research.google.com/github/arturdiel/unijui-prog-ciencia-de-dados/blob/main/analise_ipca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import pandas as pd
import numpy as np

In [29]:
url_ipca = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=csv"
url_metas = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.13521/dados?formato=csv"

ipca = pd.read_csv(url_ipca, sep=";")
metas = pd.read_csv(url_metas, sep=";")

In [30]:
# Começo e do fim da série
print("Primeiros 5 meses da série:")
print(ipca.head())
print("\nÚltimos 5 meses da série:")
print(ipca.tail())

Primeiros 5 meses da série:
         data valor
0  01/01/1980  6,62
1  01/02/1980  4,62
2  01/03/1980  6,04
3  01/04/1980  5,29
4  01/05/1980  5,70

Últimos 5 meses da série:
           data  valor
556  01/05/2026   0,58
557  01/06/2026   0,16
558  01/07/2026   0,07
559  01/08/2026  -0,32
560  01/09/2026   0,82


In [31]:
# Quantidade de registros
print("Quantidade de registros (meses):", ipca.shape[0])
print("Quantidade de atributos (colunas):", ipca.shape[1])

Quantidade de registros (meses): 561
Quantidade de atributos (colunas): 2


In [32]:
# Tipo de cada coluna
print("Tipo de cada coluna:")
print(ipca.dtypes)

Tipo de cada coluna:
data     object
valor    object
dtype: object


In [33]:
# Valores ausentes em cada coluna
print("Valores ausentes em cada coluna:")
print(ipca.isnull().sum())

Valores ausentes em cada coluna:
data     0
valor    0
dtype: int64


In [34]:
# Tabela de metas
print("Primeiros 5 anos da tabela de metas:")
print(metas.head())
print("\nQuantidade de registros (anos):", metas.shape[0])
print("Quantidade de atributos (colunas):", metas.shape[1])
print("\nTipo de cada coluna:")
print(metas.dtypes)
print("\nValores ausentes em cada coluna:")
print(metas.isnull().sum())

Primeiros 5 anos da tabela de metas:
         data valor
0  01/01/1999  8,00
1  01/01/2000  6,00
2  01/01/2001  4,00
3  01/01/2002  3,50
4  01/01/2003  4,00

Quantidade de registros (anos): 28
Quantidade de atributos (colunas): 2

Tipo de cada coluna:
data     object
valor    object
dtype: object

Valores ausentes em cada coluna:
data     0
valor    0
dtype: int64


In [35]:
# Converte o valor de texto para número
ipca["valor"] = ipca["valor"].str.replace(",", ".").astype(float)
metas["valor"] = metas["valor"].str.replace(",", ".").astype(float)

print("Tipo de cada coluna depois de converter o valor:")
print(ipca.dtypes)

Tipo de cada coluna depois de converter o valor:
data      object
valor    float64
dtype: object


In [36]:
# Converte a data de texto para data
ipca["data"] = pd.to_datetime(ipca["data"], format="%d/%m/%Y")
metas["data"] = pd.to_datetime(metas["data"], format="%d/%m/%Y")

ipca["ano"] = ipca["data"].dt.year
ipca["mes"] = ipca["data"].dt.month
metas["ano"] = metas["data"].dt.year

print("Tabela com as novas colunas de ano e mês:")
print(ipca.head())
print("\nTipo de cada coluna depois de converter a data:")
print(ipca.dtypes)

Tabela com as novas colunas de ano e mês:
        data  valor   ano  mes
0 1980-01-01   6.62  1980    1
1 1980-02-01   4.62  1980    2
2 1980-03-01   6.04  1980    3
3 1980-04-01   5.29  1980    4
4 1980-05-01   5.70  1980    5

Tipo de cada coluna depois de converter a data:
data     datetime64[ns]
valor           float64
ano               int32
mes               int32
dtype: object


In [37]:
# Confere os valores extremos
print(f"Maior inflação mensal: {ipca['valor'].max():.2f}%")
print(f"Menor inflação mensal: {ipca['valor'].min():.2f}%")

maiores_da_serie = ipca.sort_values("valor", ascending=False)
print("\nOs 5 meses de maior inflação da série inteira (%):")
print(maiores_da_serie[["data", "valor"]].head())

Maior inflação mensal: 82.39%
Menor inflação mensal: -0.68%

Os 5 meses de maior inflação da série inteira (%):
          data  valor
122 1990-03-01  82.39
121 1990-02-01  75.73
120 1990-01-01  67.55
119 1989-12-01  51.50
118 1989-11-01  47.82


In [38]:
# Separa os períodos antes e depois do Plano Real
ipca["periodo"] = np.where(ipca["data"] >= "1994-07-01", "Plano Real", "Antes do Real")

print("Quantidade de meses em cada período:")
print(ipca.groupby("periodo")["valor"].count())

Quantidade de meses em cada período:
periodo
Antes do Real    174
Plano Real       387
Name: valor, dtype: int64


In [39]:
# Quantidade de meses em cada ano
meses_por_ano = ipca.groupby("ano")["valor"].count()
print("Quantidade de meses nos últimos 5 anos da série:")
print(meses_por_ano.tail())

Quantidade de meses nos últimos 5 anos da série:
ano
2022    12
2023    12
2024    12
2025    12
2026     9
Name: valor, dtype: int64


In [40]:
# Inflação acumulada de cada ano
ipca["fator"] = 1 + ipca["valor"] / 100

anos_completos = ipca[(ipca["ano"] >= 1995) & (ipca["ano"] <= 2025)]

anual = (anos_completos.groupby("ano")["fator"].prod() - 1) * 100

anual = anual.round(2)
print("Inflação acumulada de cada ano (%):")
print(anual)

Inflação acumulada de cada ano (%):
ano
1995    22.41
1996     9.56
1997     5.22
1998     1.66
1999     8.94
2000     5.97
2001     7.67
2002    12.53
2003     9.30
2004     7.60
2005     5.69
2006     3.14
2007     4.46
2008     5.90
2009     4.31
2010     5.91
2011     6.50
2012     5.84
2013     5.91
2014     6.41
2015    10.67
2016     6.29
2017     2.95
2018     3.75
2019     4.31
2020     4.52
2021    10.06
2022     5.78
2023     4.62
2024     4.83
2025     4.26
Name: fator, dtype: float64


In [41]:
# Junta a inflação de cada ano com a meta do mesmo ano
anual = anual.reset_index()
anual = anual.rename(columns={"fator": "ipca_anual"})
metas = metas.rename(columns={"valor": "meta"})

comparacao = pd.merge(anual, metas[["ano", "meta"]], on="ano", how="left")
print("Primeiros 6 anos da tabela juntada (inflação e meta):")
print(comparacao.head(6))
print("\nValores ausentes em cada coluna depois de juntar:")
print(comparacao.isnull().sum())

Primeiros 6 anos da tabela juntada (inflação e meta):
    ano  ipca_anual  meta
0  1995       22.41   NaN
1  1996        9.56   NaN
2  1997        5.22   NaN
3  1998        1.66   NaN
4  1999        8.94   8.0
5  2000        5.97   6.0

Valores ausentes em cada coluna depois de juntar:
ano           0
ipca_anual    0
meta          4
dtype: int64


In [42]:
# Trata os valores ausentes
comparacao = comparacao.dropna()

print("Anos com meta:", comparacao.shape[0])
print("\nValores ausentes em cada coluna depois do dropna:")
print(comparacao.isnull().sum())

Anos com meta: 27

Valores ausentes em cada coluna depois do dropna:
ano           0
ipca_anual    0
meta          0
dtype: int64


In [43]:
# Inflação mensal antes e depois do Plano Real
media_por_periodo = ipca.groupby("periodo")["valor"].mean()
maior_por_periodo = ipca.groupby("periodo")["valor"].max()

print("Inflação média por mês em cada período (%):")
print(media_por_periodo.round(2))
print("\nMaior inflação em um único mês em cada período (%):")
print(maior_por_periodo.round(2))

vezes = media_por_periodo["Antes do Real"] / media_por_periodo["Plano Real"]
print(f"\nA inflação média mensal ficou {vezes:.0f} vezes menor depois do Real")

Inflação média por mês em cada período (%):
periodo
Antes do Real    16.44
Plano Real        0.57
Name: valor, dtype: float64

Maior inflação em um único mês em cada período (%):
periodo
Antes do Real    82.39
Plano Real        6.84
Name: valor, dtype: float64

A inflação média mensal ficou 29 vezes menor depois do Real


In [44]:
# Os 5 meses de maior inflação no Plano Real
real = ipca[ipca["periodo"] == "Plano Real"]

maiores_meses = real.sort_values("valor", ascending=False)
print("Os 5 meses de maior inflação no Plano Real (%):")
print(maiores_meses[["data", "valor"]].head())

Os 5 meses de maior inflação no Plano Real (%):
          data  valor
174 1994-07-01   6.84
274 2002-11-01   3.02
178 1994-11-01   2.81
184 1995-05-01   2.67
177 1994-10-01   2.62


In [45]:
# Inflação média de cada mês do ano (1995 a 2025)
MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]

media_por_mes = anos_completos.groupby("mes")["valor"].mean()
media_por_mes.index = MESES

print("Inflação média de cada mês do ano (%):")
print(media_por_mes.round(2))
print(f"\nMês com maior inflação média: {media_por_mes.idxmax()} ({media_por_mes.max():.2f}%)")
print(f"Mês com menor inflação média: {media_por_mes.idxmin()} ({media_por_mes.min():.2f}%)")

Inflação média de cada mês do ano (%):
jan    0.71
fev    0.69
mar    0.60
abr    0.61
mai    0.49
jun    0.39
jul    0.49
ago    0.30
set    0.36
out    0.54
nov    0.57
dez    0.68
Name: valor, dtype: float64

Mês com maior inflação média: jan (0.71%)
Mês com menor inflação média: ago (0.30%)


In [46]:
# Meses com deflação (inflação negativa) no Plano Real
deflacao = real[real["valor"] < 0]

print("Meses com deflação:", len(deflacao), "de", len(real))
print("\nLista dos meses com deflação (%):")
print(deflacao[["data", "valor"]])

print("\nQuantidade de deflações em cada mês do ano (1 = janeiro, 12 = dezembro):")
print(deflacao.groupby("mes")["valor"].count())

Meses com deflação: 21 de 387

Lista dos meses com deflação (%):
          data  valor
211 1997-08-01  -0.02
222 1998-07-01  -0.12
223 1998-08-01  -0.51
224 1998-09-01  -0.22
226 1998-11-01  -0.12
281 2003-06-01  -0.15
305 2005-06-01  -0.02
317 2006-06-01  -0.21
449 2017-06-01  -0.23
463 2018-08-01  -0.09
466 2018-11-01  -0.21
476 2019-09-01  -0.04
483 2020-04-01  -0.31
484 2020-05-01  -0.38
510 2022-07-01  -0.68
511 2022-08-01  -0.36
512 2022-09-01  -0.29
521 2023-06-01  -0.08
535 2024-08-01  -0.02
547 2025-08-01  -0.11
559 2026-08-01  -0.32

Quantidade de deflações em cada mês do ano (1 = janeiro, 12 = dezembro):
mes
4     1
5     1
6     5
7     2
8     7
9     3
11    2
Name: valor, dtype: int64


In [47]:
# Os 5 anos de maior e de menor inflação no Plano Real (1995 a 2025)
print("Os 5 anos de maior inflação (%):")
print(anual.sort_values("ipca_anual", ascending=False).head())

print("\nOs 5 anos de menor inflação (%):")
print(anual.sort_values("ipca_anual").head())

Os 5 anos de maior inflação (%):
     ano  ipca_anual
0   1995       22.41
7   2002       12.53
20  2015       10.67
26  2021       10.06
1   1996        9.56

Os 5 anos de menor inflação (%):
     ano  ipca_anual
3   1998        1.66
22  2017        2.95
11  2006        3.14
23  2018        3.75
30  2025        4.26


In [48]:
# Comparando a inflação de cada ano com a meta
comparacao["diferenca"] = comparacao["ipca_anual"] - comparacao["meta"]

acima_da_meta = comparacao[comparacao["ipca_anual"] > comparacao["meta"]]
print("Anos acima da meta:", len(acima_da_meta), "de", len(comparacao))

na_meta_ou_abaixo = comparacao[comparacao["ipca_anual"] <= comparacao["meta"]]
print("\nAnos em que a inflação ficou na meta ou abaixo dela (%):")
print(na_meta_ou_abaixo[["ano", "ipca_anual", "meta"]])

Anos acima da meta: 21 de 27

Anos em que a inflação ficou na meta ou abaixo dela (%):
     ano  ipca_anual  meta
5   2000        5.97   6.0
11  2006        3.14   4.5
12  2007        4.46   4.5
14  2009        4.31   4.5
22  2017        2.95   4.5
23  2018        3.75   4.5


In [49]:
# Os 5 anos que ficaram mais acima da meta
mais_longe_da_meta = comparacao.sort_values("diferenca", ascending=False)
print("Os 5 anos que ficaram mais acima da meta (diferença em pontos percentuais):")
print(mais_longe_da_meta.head().round(2))

Os 5 anos que ficaram mais acima da meta (diferença em pontos percentuais):
     ano  ipca_anual  meta  diferenca
7   2002       12.53  3.50       9.03
26  2021       10.06  3.75       6.31
20  2015       10.67  4.50       6.17
8   2003        9.30  4.00       5.30
6   2001        7.67  4.00       3.67


In [50]:
# Resumo
ano_maior = anual.sort_values("ipca_anual", ascending=False).iloc[0]
ano_menor = anual.sort_values("ipca_anual").iloc[0]

print("RESUMO DA ANÁLISE")
print(f"Meses analisados: {ipca.shape[0]}")
print(f"Inflação média mensal antes do Real: {media_por_periodo['Antes do Real']:.2f}%")
print(f"Inflação média mensal no Plano Real: {media_por_periodo['Plano Real']:.2f}%")
print(f"Mês do ano com maior inflação média: {media_por_mes.idxmax()} ({media_por_mes.max():.2f}%)")
print(f"Meses com deflação no Plano Real: {len(deflacao)}")
print(f"Ano de maior inflação: {ano_maior['ano']:.0f} ({ano_maior['ipca_anual']:.2f}%)")
print(f"Ano de menor inflação: {ano_menor['ano']:.0f} ({ano_menor['ipca_anual']:.2f}%)")
print(f"Anos acima da meta: {len(acima_da_meta)} de {len(comparacao)}")

RESUMO DA ANÁLISE
Meses analisados: 561
Inflação média mensal antes do Real: 16.44%
Inflação média mensal no Plano Real: 0.57%
Mês do ano com maior inflação média: jan (0.71%)
Meses com deflação no Plano Real: 21
Ano de maior inflação: 1995 (22.41%)
Ano de menor inflação: 1998 (1.66%)
Anos acima da meta: 21 de 27


In [51]:
comparacao.round(2).to_csv("ipca_anual_vs_meta.csv", index=False)
print("Arquivo ipca_anual_vs_meta.csv salvo com sucesso!")

Arquivo ipca_anual_vs_meta.csv salvo com sucesso!


In [52]:
!git add .
!git commit -m "Arquivos de analise do ipca"
!git push

[main d56a68f] Arquivos de analise do ipca
 1 file changed, 28 insertions(+)
 create mode 100644 ipca_anual_vs_meta.csv
Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 558 bytes | 558.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/arturdiel/unijui-prog-ciencia-de-dados
   2ec973e..d56a68f  main -> main
